## Traffic Demand Prediction Model
Loading the dependencies for the task.

In [2]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

### Load the Datasets
Reading `train.csv` and `test.csv`.

In [3]:
train = pd.read_csv("./dataset/train.csv")
test = pd.read_csv("./dataset/test.csv")

print("Train size:", train.shape)
print("Test size:", test.shape)

Train size: (77299, 11)
Test size: (41778, 10)


### Feature Engineering
Parsing time fields, deriving `time_in_mins`, `day_of_week` and setting default categorical/numerical imputations to prevent target leaks.

In [4]:
def preprocess(df):
    df_new = df.copy()
    
    df_new['hour'] = df_new['timestamp'].apply(lambda x: int(x.split(':')[0]))
    df_new['minute'] = df_new['timestamp'].apply(lambda x: int(x.split(':')[1]))
    df_new['time_in_mins'] = df_new['hour'] * 60 + df_new['minute']
    
    for col in ['RoadType', 'LargeVehicles', 'Landmarks', 'Weather']:
        if col in df_new.columns:
            df_new[col] = df_new[col].fillna('Unknown')
    
    temp_mean = df_new['Temperature'].mean()
    lanes_mean = df_new['NumberofLanes'].mean()
    df_new['Temperature'] = df_new['Temperature'].fillna(temp_mean)
    df_new['NumberofLanes'] = df_new['NumberofLanes'].fillna(lanes_mean)
    
    df_new['day_of_week'] = df_new['day'] % 7
    df_new['is_weekend'] = (df_new['day_of_week'] >= 5).astype(int)
    
    return df_new

X_train = preprocess(train)
X_test = preprocess(test)

y_train = X_train['demand']
X_train = X_train.drop(['demand', 'Index', 'timestamp'], axis=1)

X_test_ids = X_test['Index']
X_test = X_test.drop(['Index', 'timestamp'], axis=1)

### K-Fold Cross Validation Setup
Training CatBoost using 5 independent folds to maximize evaluation `R2` metrics.

In [5]:
categorical_features = ['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']

kf = KFold(n_splits=5, shuffle=True, random_state=42)
test_preds = np.zeros(len(X_test))
oof_preds = np.zeros(len(X_train))

print("Starting 5-Fold Training...")
for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
    print(f"--- Fold {fold+1} ---")
    X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]
    
    model = CatBoostRegressor(
        iterations=4000,
        learning_rate=0.06,
        depth=8,
        l2_leaf_reg=4,
        loss_function='RMSE',
        eval_metric='R2',
        random_seed=42 + fold,
        early_stopping_rounds=150,
        verbose=1000
    )
    
    model.fit(
        X_tr, y_tr,
        eval_set=(X_val, y_val),
        cat_features=categorical_features,
        use_best_model=True
    )
    
    oof_preds[val_idx] = model.predict(X_val)
    test_preds += model.predict(X_test) / kf.n_splits

oof_r2 = r2_score(y_train, oof_preds)
print(f"\nOverall Out-Of-Fold R2 Score: {oof_r2:.6f}")

Starting 5-Fold Training...
--- Fold 1 ---
0:	learn: 0.0862960	test: 0.0882513	best: 0.0882513 (0)	total: 73.8ms	remaining: 4m 55s
1000:	learn: 0.9521648	test: 0.9415249	best: 0.9415374 (999)	total: 20.3s	remaining: 1m
2000:	learn: 0.9621416	test: 0.9447164	best: 0.9447300 (1991)	total: 40s	remaining: 39.9s
3000:	learn: 0.9675624	test: 0.9458457	best: 0.9458489 (2871)	total: 1m	remaining: 20.1s
3999:	learn: 0.9716014	test: 0.9463754	best: 0.9463765 (3990)	total: 1m 20s	remaining: 0us

bestTest = 0.9463764824
bestIteration = 3990

Shrink model to first 3991 iterations.
--- Fold 2 ---
0:	learn: 0.0865660	test: 0.0856848	best: 0.0856848 (0)	total: 21.5ms	remaining: 1m 25s
1000:	learn: 0.9519979	test: 0.9410782	best: 0.9410782 (1000)	total: 19.6s	remaining: 58.8s
2000:	learn: 0.9624031	test: 0.9447576	best: 0.9447576 (2000)	total: 42.7s	remaining: 42.6s
3000:	learn: 0.9681386	test: 0.9459951	best: 0.9459951 (3000)	total: 1m 4s	remaining: 21.5s
3999:	learn: 0.9722844	test: 0.9466464	best: 0

### Generate Submission File
Clipping variables to limits `(0, None)` as demand can't be negative, and saving.

In [6]:
submission = pd.DataFrame({
    'Index': X_test_ids,
    'demand': np.clip(test_preds, 0, None)
})

submission.to_csv("submission.csv", index=False)
print("Submission saved!")

Submission saved!
